# Milestone 0 — Data Preparation & Exploratory Data Analysis

**Smart Product Intelligence Capstone** — Khazar University

This notebook covers M0:
- Loading and cleaning Amazon Reviews 2023 (Toys & Games)
- Constructing **product-level** train/val/test splits (no leakage)
- Reporting rating distribution, price, review length, missing data
- Stretch goal: documenting data-quality issues

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 100})

PROJECT_ROOT = '/content/drive/MyDrive/smart-product-intelligence'

## 1. Mount Drive and load cached subset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
products = pd.read_csv(os.path.join(DATA_DIR, 'products.csv'))
reviews = pd.read_csv(os.path.join(DATA_DIR, 'reviews.csv'))
image_index = pd.read_csv(os.path.join(DATA_DIR, 'image_index.csv'))

print(f'Products: {len(products):,}')
print(f'Reviews:  {len(reviews):,}')
print(f'Cached images: {len(image_index):,}')

## 2. Verify product-level split (no leakage)

The brief warns that splitting by review would leak information across sets.
We split by `parent_asin` and verify intersections are exactly zero.

In [ ]:
train_ids = set(products[products['split']=='train']['parent_asin'])
val_ids = set(products[products['split']=='val']['parent_asin'])
test_ids = set(products[products['split']=='test']['parent_asin'])

print(f'Train: {len(train_ids):,} products')
print(f'Val:   {len(val_ids):,} products')
print(f'Test:  {len(test_ids):,} products')

print(f'\nIntersections (must be 0):')
print(f'  train ∩ val   = {len(train_ids & val_ids)}')
print(f'  train ∩ test  = {len(train_ids & test_ids)}')
print(f'  val   ∩ test  = {len(val_ids & test_ids)}')

assert len(train_ids & val_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(val_ids & test_ids) == 0
print('\n✅ No data leakage')

## 3. Rating distribution — the imbalance that motivates macro-F1

5-star reviews dominate. We report macro-F1 in every classification task.

In [ ]:
rating_counts = reviews['rating'].value_counts().sort_index()
total = rating_counts.sum()

print('Rating distribution:')
for r, c in rating_counts.items():
    print(f'  {int(r)} stars: {c:>6,} ({100*c/total:5.1f}%)')

print(f'\n5-star share: {100*rating_counts[5]/total:.1f}%')

## 4. Price, review length, and missing-data report

In [ ]:
prices = products['price_clean'].dropna()
prices = prices[(prices > 0) & (prices < 200)]

print(f'Price (filled): mean=${prices.mean():.2f}, median=${prices.median():.2f}')
print(f'Missing prices: {100*products["price_clean"].isna().sum()/len(products):.1f}%')

review_lens = reviews['text_length']
print(f'\nReview length: mean={review_lens.mean():.0f}, median={review_lens.median():.0f} chars')
print(f'Image coverage: {100*len(image_index)/len(products):.1f}% of products')

## 5. Overview figure

In [ ]:
from IPython.display import Image as IPyImage
IPyImage(filename=os.path.join(PROJECT_ROOT, 'figures', '00_eda_overview.png'))

## 6. Stretch goal — data-quality issues

1. **Broken image links.** Cached during M0; never fetched live during training.
2. **Missing prices.** 43% of products. Kept with a `price_was_missing` flag.
3. **Rating imbalance.** 66% 5-star. Handled with macro-F1 and class weighting.
4. **Sparse descriptions.** Some products are title+image only.

## 7. Summary

| Quantity | Value |
|---|---|
| Products | 15,000 |
| Reviews | 75,000 |
| Cached images | 7,996 |
| Splits (train/val/test) | 12,000 / 1,500 / 1,500 |
| Leakage between splits | 0 |
| 5-star share | ~66% |
| Missing prices | ~43% |